# L12 — RFQ & Probabilidad de Cierre

Trabajamos con **800 RFQs sintéticos y reproducibles** de bonos Tesoro español. El objetivo es construir un modelo que prediga la probabilidad de cerrar una operación dado el spread que cotizamos.

---

## Tiers de ejercicios

| Tier | Ejercicios | Cuándo |
|------|-----------|--------|
| **Núcleo** | 1–5 | Completa en clase |
| **Si vamos bien** | 6–7 | Si el ritmo lo permite |
| **Bonus / casa** | 8–10 | Tarea o para los más rápidos |

---

## Ejercicio 0 — Reflexión inicial (sin código)

Antes de abrir el dataset, responde mentalmente estas preguntas:

1. Un RFQ termina con `won=0` y `closed_away=0`. ¿Sabes si el cliente cruzó la operación o no? ¿Por qué es un problema esto para entrenar un modelo?

2. Tenemos spreads en **precio** (ej. `0.0087` de diferencia en precio por 100 nominal). ¿Por qué es difícil comparar ese número entre un bono a 2 años y uno a 30 años?

3. Queremos predecir `P(cierre | spread)`. ¿Por qué esperamos que esa función sea **decreciente** en el spread? ¿Siempre tiene que serlo?

4. Si encontramos el spread óptimo `s*` que maximiza `E[Revenue]`, ¿crees que `s*` depende del nominal de la operación? Razona tu respuesta.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from sklearn.linear_model import LogisticRegression
from scipy.special import expit  # sigmoid
from scipy.optimize import minimize_scalar

# Plot theme
plt.rcParams.update({
    'figure.facecolor': '#09090b',
    'axes.facecolor': '#18181b',
    'axes.edgecolor': '#27272a',
    'axes.labelcolor': '#a1a1aa',
    'xtick.color': '#71717a',
    'ytick.color': '#71717a',
    'text.color': '#f4f4f5',
    'grid.color': '#27272a',
    'grid.linewidth': 0.5,
})
TIER_COLOR = {1: '#22d3ee', 2: '#fbbf24', 3: '#f87171'}

print('Setup completado ✓')

---
## Ejercicio 1 — Carga y exploración del dataset
*(Núcleo)*

Carga `../data/rfq_dataset.csv` en un DataFrame `df`. Luego:
- Comprueba que tiene 800 filas y 19 columnas.
- Imprime las primeras 3 filas.
- Cuenta cuántos RFQs hay por tier.

In [ ]:
# Tu código aquí
df = None
pass

In [ ]:
# ── Validación E1 ──────────────────────────────────────────────────
assert 'df' in globals() and df is not None, "Define df antes de continuar"
assert df.shape == (800, 19), f"Esperaba (800, 19), tienes {df.shape}. ¿Cargaste el CSV correcto?"
expected_cols = {'rfq_id','timestamp','bond_name','tenor','coupon','mid_price_cbbt',
                 'spread_price_cbbt','best_bid_mts','best_ask_mts','ytm','dv01',
                 'tier','side','volume_meur','num_dealers','quoted_price',
                 'won','closed_away','cover_price'}
assert expected_cols == set(df.columns), f"Columnas inesperadas. Revisa el CSV."
assert df['won'].isin([0,1]).all(), "'won' debe ser 0 o 1"
assert set(df['tier'].unique()) == {1,2,3}, "'tier' debe tener valores 1, 2, 3"
print("✓ E1 correcto — 800 RFQs, 19 columnas, columnas OK")

In [ ]:
# ── Solución E1 ───────────────────────────────────────────────────
df = pd.read_csv('../data/rfq_dataset.csv')
print(f"Shape: {df.shape}")
print(df.head(3).to_string())
print("\nRFQs por tier:")
print(df['tier'].value_counts().sort_index())

---
## Ejercicio 2 — La feature `spread_bp`
*(Núcleo)*

El spread cotizado está en **precio** (diferencia en precio por 100 nominal), pero varía enormemente con el tenor. Necesitamos normalizarlo por el DV01 para que sea comparable entre bonos.

**Fórmula:**
```
spread_bp = (quoted_price - mid_price_cbbt) / dv01   # si side == 1 (BUY)
spread_bp = (mid_price_cbbt - quoted_price) / dv01   # si side == 2 (SELL)
```

Añade la columna `spread_bp` al DataFrame `df`.

In [ ]:
# Tu código aquí
df['spread_bp'] = None
pass

In [ ]:
# ── Validación E2 ──────────────────────────────────────────────────
assert 'spread_bp' in df.columns, "Falta la columna 'spread_bp'"
assert df['spread_bp'].notna().all(), "'spread_bp' no debe tener NaN"
# Mean spread_bp by tier (pre-computed)
for tier, expected_mean in [(1, 0.0761), (2, 0.4136), (3, 0.6600)]:
    actual = df.loc[df['tier']==tier, 'spread_bp'].mean()
    assert abs(actual - expected_mean) < 0.005, \
        f"T{tier}: media esperada ≈{expected_mean}, tienes {actual:.4f}. Revisa la fórmula del spread."
print("✓ E2 correcto — spread_bp calculado correctamente")

In [ ]:
# ── Solución E2 ───────────────────────────────────────────────────
df['spread_bp'] = np.where(
    df['side'] == 1,
    (df['quoted_price'] - df['mid_price_cbbt']) / df['dv01'],
    (df['mid_price_cbbt'] - df['quoted_price']) / df['dv01']
)
print("spread_bp media por tier:")
for t in [1,2,3]:
    m = df['tier']==t
    print(f"  T{t}: {df.loc[m,'spread_bp'].mean():.4f} pb (min={df.loc[m,'spread_bp'].min():.3f}, max={df.loc[m,'spread_bp'].max():.3f})")

---
## Ejercicio 3 — Filtrar: construir `df_model`
*(Núcleo)*

Los RFQs con `won=0` y `closed_away=0` son ambiguos: no sabemos si fue *price discovery* o si el cliente cruzó con alguien y no tenemos el dato.

**Construye `df_model`** manteniendo solo los RFQs donde:
- `won == 1` (ganamos), **o**
- `closed_away == 1` (otro dealer ganó).

Resetea el índice y verifica el shape resultante.

In [ ]:
# Tu código aquí
df_model = None
pass

In [ ]:
# ── Validación E3 ──────────────────────────────────────────────────
assert 'df_model' in globals() and df_model is not None, "Define df_model"
assert df_model.shape[0] == 557, \
    f"Esperaba 557 filas, tienes {df_model.shape[0]}. ¿Filtraste correctamente?"
assert (df_model['won'] | df_model['closed_away']).all(), \
    "Hay filas en df_model donde won=0 Y closed_away=0 — revisa el filtro"
assert (df_model['won'].sum() == 65), \
    f"Debería haber 65 wins, tienes {df_model['won'].sum()}"
print(f"✓ E3 correcto — df_model: {df_model.shape[0]} RFQs (65 won + 492 closed_away)")

In [ ]:
# ── Solución E3 ───────────────────────────────────────────────────
df_model = df[
    (df['won'] == 1) | (df['closed_away'] == 1)
].reset_index(drop=True)

print(f"df_model shape: {df_model.shape}")
print(f"\nWon rate por tier (corregida):")
for t in [1,2,3]:
    m = df_model['tier']==t
    print(f"  T{t}: {df_model.loc[m,'won'].mean():.1%}  (n={m.sum()})")

---
## Ejercicio 4 — Regresión logística por tier
*(Núcleo)*

Entrena un modelo de regresión logística (1 feature: `spread_bp`) para cada tier usando `df_model`.

Guarda los resultados en el diccionario `models` con la estructura:
```python
models = {
    1: {'b0': float, 'b1': float, 'model': LogisticRegression},
    2: {'b0': float, 'b1': float, 'model': LogisticRegression},
    3: {'b0': float, 'b1': float, 'model': LogisticRegression},
}
```

Imprime `b0` y `b1` por tier. ¿Qué observas en el signo de `b1` para Tier 3?

In [ ]:
# Tu código aquí
models = {}
pass

In [ ]:
# ── Validación E4 ──────────────────────────────────────────────────
assert 'models' in globals() and len(models) == 3, "models debe tener 3 tiers"
expected = {
    1: (-1.153594, -1.164688),
    2: (-1.895052, -0.336315),
    3: (-3.082927,  0.677225),
}
for t, (eb0, eb1) in expected.items():
    assert t in models, f"Falta tier {t} en models"
    assert 'b0' in models[t] and 'b1' in models[t], f"Tier {t}: falta 'b0' o 'b1'"
    assert abs(models[t]['b0'] - eb0) < 0.001, \
        f"T{t}: b0 esperado ≈{eb0:.4f}, tienes {models[t]['b0']:.4f}"
    assert abs(models[t]['b1'] - eb1) < 0.001, \
        f"T{t}: b1 esperado ≈{eb1:.4f}, tienes {models[t]['b1']:.4f}"
print("✓ E4 correcto — modelos logísticos correctos")
print(f"  T3 b1 = {models[3]['b1']:.4f} > 0 → ¡caso especial!")

In [ ]:
# ── Solución E4 ───────────────────────────────────────────────────
models = {}
for t in [1, 2, 3]:
    mask = df_model['tier'] == t
    X = df_model.loc[mask, 'spread_bp'].values.reshape(-1, 1)
    y = df_model.loc[mask, 'won'].values
    lr = LogisticRegression(random_state=42)
    lr.fit(X, y)
    models[t] = {
        'b0': lr.intercept_[0],
        'b1': lr.coef_[0][0],
        'model': lr
    }
    print(f"T{t}: b0 = {lr.intercept_[0]:.6f}  b1 = {lr.coef_[0][0]:+.6f}")

---
## Ejercicio 5 — `predict_close_probability(tier, spread_bp)`
*(Núcleo)*

Escribe la función `predict_close_probability(tier, spread_bp)` que devuelve un float entre 0 y 1.

Usa los parámetros del modelo que acabas de entrenar. La función debe funcionar para cualquier tier (1, 2 o 3).

Después, visualiza las tres curvas en un mismo gráfico.

In [ ]:
# Tu código aquí
def predict_close_probability(tier, spread_bp):
    pass

In [ ]:
# ── Validación E5 ──────────────────────────────────────────────────
assert callable(predict_close_probability), "predict_close_probability debe ser una función"
cases = [
    (1, 0.5,  0.14983),
    (1, 0.0,  0.23983),
    (2, 0.4,  0.11613),
    (3, 1.0,  0.08274),
]
for tier, s, expected in cases:
    result = predict_close_probability(tier, s)
    assert isinstance(result, float), f"Debe devolver float, devuelve {type(result)}"
    assert abs(result - expected) < 0.001, \
        f"predict_close_probability({tier}, {s}): esperado {expected:.5f}, tienes {result:.5f}"
# Monotone check for T1 and T2
for t in [1, 2]:
    vals = [predict_close_probability(t, s) for s in [0.0, 0.3, 0.6, 0.9]]
    assert vals == sorted(vals, reverse=True), \
        f"T{t}: la curva debe ser decreciente en spread"
print("✓ E5 correcto — predict_close_probability funciona")

In [ ]:
# ── Solución E5 ───────────────────────────────────────────────────
def predict_close_probability(tier, spread_bp):
    b0 = models[tier]['b0']
    b1 = models[tier]['b1']
    return float(expit(b0 + b1 * spread_bp))

# Visualización
fig, ax = plt.subplots(figsize=(9, 5))
s_range = np.linspace(-0.3, 1.3, 300)

for t in [1, 2, 3]:
    probs = [predict_close_probability(t, s) for s in s_range]
    ax.plot(s_range, probs, color=TIER_COLOR[t], linewidth=2.5, label=f'Tier {t}')

ax.set_xlabel('Spread cotizado (pb)')
ax.set_ylabel('P(cierre)')
ax.set_title('Curvas de cierre por tier', color='#f4f4f5')
ax.legend(framealpha=0.2)
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
plt.tight_layout()
plt.show()

---
## Ejercicio 6 — Ingreso esperado `expected_revenue(tier, spread_bp, Q=1.0)`
*(Si vamos bien)*

El ingreso esperado de cotizar con spread `s` en un RFQ de nominal `Q` es:

$$E[\text{Rev}] = P(\text{cierre} \mid s) \times s \times Q$$

Escribe `expected_revenue(tier, spread_bp, Q=1.0)`. Visualiza la curva para los tres tiers con Q=1.

**Pregunta:** ¿Qué pasa con la forma de la curva para Tier 3?

In [ ]:
# Tu código aquí
def expected_revenue(tier, spread_bp, Q=1.0):
    pass

In [ ]:
# ── Validación E6 ──────────────────────────────────────────────────
assert callable(expected_revenue), "expected_revenue debe ser una función"
cases = [
    (1, 0.5,  1.0, 0.074915),
    (1, 0.948361, 1.0, 0.089762),
    (2, 0.4,  2.0, 0.092906),
    (3, 1.0,  1.0, 0.082739),
]
for tier, s, Q, expected in cases:
    result = expected_revenue(tier, s, Q)
    assert abs(result - expected) < 0.001, \
        f"expected_revenue({tier}, {s}, {Q}): esperado {expected:.6f}, tienes {result:.6f}"
# E[Rev] at s=0 should be 0
for t in [1,2,3]:
    assert abs(expected_revenue(t, 0.0)) < 1e-9, \
        f"T{t}: E[Rev] en s=0 debe ser 0"
print("✓ E6 correcto — expected_revenue OK")

In [ ]:
# ── Solución E6 ───────────────────────────────────────────────────
def expected_revenue(tier, spread_bp, Q=1.0):
    return predict_close_probability(tier, spread_bp) * spread_bp * Q

# Visualización
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
s_range = np.linspace(0.0, 1.5, 300)

for i, t in enumerate([1, 2, 3]):
    ax = axes[i]
    revs = [expected_revenue(t, s) for s in s_range]
    ax.plot(s_range, revs, color=TIER_COLOR[t], linewidth=2.5)
    ax.set_xlabel('Spread (pb)')
    ax.set_ylabel('E[Rev]')
    ax.set_title(f'Tier {t}', color=TIER_COLOR[t])
    ax.grid(True, alpha=0.3)

fig.suptitle('E[Revenue] = P(cierre) × spread × Q  (Q=1)', color='#f4f4f5')
plt.tight_layout()
plt.show()

print("Nota: Tier 3 crece de forma monótona — no hay máximo interior (b1 > 0).")

---
## Ejercicio 7 — Spread óptimo `optimal_spread(tier, Q=1.0)`
*(Si vamos bien)*

Usa `scipy.optimize.minimize_scalar` para encontrar el spread que **maximiza** `expected_revenue`.

Devuelve `s*` en puntos básicos. Busca en el intervalo `[0.01, 5.0]`.

**Reflexión:** ¿Cambia `s*` al cambiar `Q`? Pruébalo antes de validar.

In [ ]:
# Tu código aquí
def optimal_spread(tier, Q=1.0):
    pass

In [ ]:
# ── Validación E7 ──────────────────────────────────────────────────
assert callable(optimal_spread), "optimal_spread debe ser una función"
cases_s = [(1, 1.0, 0.9484), (2, 1.0, 3.1294)]
for tier, Q, expected_s in cases_s:
    result = optimal_spread(tier, Q)
    assert abs(result - expected_s) < 0.01, \
        f"optimal_spread({tier}): esperado ≈{expected_s:.4f}, tienes {result:.4f}"
# Q invariance: s* should not change with Q
for t in [1, 2]:
    s1 = optimal_spread(t, Q=1.0)
    s10 = optimal_spread(t, Q=10.0)
    assert abs(s1 - s10) < 0.01, \
        f"T{t}: s* debería ser independiente de Q. s*(Q=1)={s1:.4f}, s*(Q=10)={s10:.4f}"
print("✓ E7 correcto — optimal_spread OK, Q no afecta s*")
for t in [1,2,3]:
    s = optimal_spread(t)
    print(f"  T{t}: s* = {s:.4f} pb  →  P(cierre) = {predict_close_probability(t, s):.2%}")

In [ ]:
# ── Solución E7 ───────────────────────────────────────────────────
def optimal_spread(tier, Q=1.0):
    result = minimize_scalar(
        lambda s: -expected_revenue(tier, s, Q),
        bounds=(0.01, 5.0),
        method='bounded'
    )
    return result.x

print("Spread óptimo por tier:")
for t in [1,2,3]:
    s = optimal_spread(t)
    p = predict_close_probability(t, s)
    rev = expected_revenue(t, s)
    print(f"  T{t}: s* = {s:.4f} pb  |  P = {p:.2%}  |  E[Rev] = {rev:.5f}")

---
## Ejercicio 8 — Validar con gráfico: P(s) y E[Rev] juntos
*(Bonus / casa)*

Para **Tier 1**, crea una figura con dos subplots lado a lado:
- **Izquierdo:** Curva `P(cierre | s)` con una línea vertical en `s*`.
- **Derecho:** Curva `E[Rev]` con una línea vertical en `s*` y anotación del valor máximo.

Añade una segunda figura igual para Tier 2.

**Bonus:** Para Tier 3, muestra que la curva E[Rev] no tiene un máximo interior y explica por qué.

In [ ]:
# Tu código aquí
pass

In [ ]:
# ── Solución E8 ───────────────────────────────────────────────────
def plot_tier_optimization(tier, ax_prob, ax_rev, s_max=1.8):
    color = TIER_COLOR[tier]
    s_range = np.linspace(0.001, s_max, 400)
    probs = [predict_close_probability(tier, s) for s in s_range]
    revs  = [expected_revenue(tier, s) for s in s_range]
    s_opt = optimal_spread(tier)
    p_opt = predict_close_probability(tier, s_opt)
    rev_opt = expected_revenue(tier, s_opt)

    # Prob chart
    ax_prob.plot(s_range, [p*100 for p in probs], color=color, linewidth=2)
    ax_prob.axvline(s_opt, color='white', linestyle='--', alpha=0.6, linewidth=1.5)
    ax_prob.scatter([s_opt], [p_opt*100], color='white', zorder=5, s=60)
    ax_prob.set_xlabel('Spread (pb)')
    ax_prob.set_ylabel('P(cierre) %')
    ax_prob.set_title(f'Tier {tier} — P(cierre | s)', color=color)
    ax_prob.grid(True, alpha=0.3)

    # Rev chart
    ax_rev.plot(s_range, revs, color='#a78bfa', linewidth=2)
    ax_rev.axvline(s_opt, color='white', linestyle='--', alpha=0.6, linewidth=1.5,
                   label=f's* = {s_opt:.3f} pb')
    ax_rev.scatter([s_opt], [rev_opt], color='#a78bfa', zorder=5, s=80)
    ax_rev.annotate(f'max = {rev_opt:.4f}',
                    xy=(s_opt, rev_opt), xytext=(s_opt + 0.15, rev_opt * 0.85),
                    color='white', fontsize=9,
                    arrowprops=dict(arrowstyle='->', color='white', lw=1))
    ax_rev.set_xlabel('Spread (pb)')
    ax_rev.set_ylabel('E[Rev]')
    ax_rev.set_title(f'Tier {tier} — E[Rev] = P(s)·s', color='#a78bfa')
    ax_rev.legend(framealpha=0.2)
    ax_rev.grid(True, alpha=0.3)

for tier, s_max in [(1, 1.8), (2, 4.5)]:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    plot_tier_optimization(tier, ax1, ax2, s_max=s_max)
    plt.tight_layout()
    plt.show()

# Tier 3 — monotone increasing
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
color3 = TIER_COLOR[3]
s_range3 = np.linspace(0.001, 1.5, 400)
probs3 = [predict_close_probability(3, s) for s in s_range3]
revs3  = [expected_revenue(3, s) for s in s_range3]
ax1.plot(s_range3, [p*100 for p in probs3], color=color3, linewidth=2)
ax1.set_title('Tier 3 — P(cierre | s)  [β₁ > 0]', color=color3)
ax1.set_xlabel('Spread (pb)'); ax1.set_ylabel('P(cierre) %'); ax1.grid(True, alpha=0.3)
ax2.plot(s_range3, revs3, color='#f87171', linewidth=2)
ax2.set_title('Tier 3 — E[Rev] sin máximo interior', color='#f87171')
ax2.set_xlabel('Spread (pb)'); ax2.set_ylabel('E[Rev]'); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("Tier 3: β₁ > 0 → E[Rev] monótonamente creciente → cotizar al límite del mercado")

---
## Ejercicio 9 — Modelo con dos features: `spread_bp` + `num_dealers`
*(Bonus / casa)*

El número de dealers (competidores) debería afectar negativamente la probabilidad de cierre.

Entrena un modelo logístico para **Tier 2** y **Tier 3** usando `[spread_bp, num_dealers]` como features.

Guarda en `models2feat` con la misma estructura que `models` pero añadiendo `'b_dealers'`.

**Pregunta:** ¿Cuánto cambia la probabilidad de cierre al pasar de 3 dealers a 10 dealers para T2, dado `spread_bp = 0.4`?

In [ ]:
# Tu código aquí
models2feat = {}
pass

In [ ]:
# ── Validación E9 ──────────────────────────────────────────────────
assert 'models2feat' in globals() and len(models2feat) >= 2, "models2feat debe tener T2 y T3"
expected_2feat = {
    2: {'b0': 1.3188, 'b_spread': -0.2098, 'b_dealers': -1.0489},
    3: {'b0': 2.3720, 'b_spread': -0.0415, 'b_dealers': -1.8606},
}
for t, exp in expected_2feat.items():
    assert t in models2feat, f"Falta tier {t} en models2feat"
    for key, eval in exp.items():
        actual = models2feat[t][key]
        assert abs(actual - eval) < 0.01, \
            f"T{t} {key}: esperado ≈{eval:.4f}, tienes {actual:.4f}"
# Dealers effect: P(close) should decrease as dealers increase
b0 = models2feat[2]['b0']
bs = models2feat[2]['b_spread']
bd = models2feat[2]['b_dealers']
p3  = float(expit(b0 + bs * 0.4 + bd * 3))
p10 = float(expit(b0 + bs * 0.4 + bd * 10))
assert p3 > p10, "T2: más dealers → menor prob. de cierre"
print(f"✓ E9 correcto — modelos 2 features")
print(f"  T2 con spread=0.4pb: P(3 dealers)={p3:.1%} vs P(10 dealers)={p10:.1%}")

In [ ]:
# ── Solución E9 ───────────────────────────────────────────────────
models2feat = {}
for t in [2, 3]:
    mask = df_model['tier'] == t
    X = df_model.loc[mask, ['spread_bp', 'num_dealers']].values
    y = df_model.loc[mask, 'won'].values
    lr = LogisticRegression(random_state=42, max_iter=1000)
    lr.fit(X, y)
    models2feat[t] = {
        'b0':       lr.intercept_[0],
        'b_spread': lr.coef_[0][0],
        'b_dealers':lr.coef_[0][1],
        'model':    lr
    }
    print(f"T{t}: b0={lr.intercept_[0]:.4f}  "
          f"b_spread={lr.coef_[0][0]:+.4f}  "
          f"b_dealers={lr.coef_[0][1]:+.4f}")

# Impact of dealers on T2
b0, bs, bd = models2feat[2]['b0'], models2feat[2]['b_spread'], models2feat[2]['b_dealers']
for nd in [1, 3, 5, 10, 15]:
    p = expit(b0 + bs * 0.4 + bd * nd)
    print(f"  T2 spread=0.4pb, {nd:2d} dealers → P(cierre) = {p:.1%}")

---
## Ejercicio 10 — Módulo `rfq_model.py`
*(Bonus / casa)*

Empaqueta todo en un módulo reutilizable. Crea el archivo `rfq_model.py` en la carpeta `data/` con:

```python
class RFQModel:
    def fit(self, df_model):
        """Entrena un modelo logístico por tier."""
        ...

    def predict_close_probability(self, tier, spread_bp):
        """Devuelve P(cierre | tier, spread_bp)."""
        ...

    def optimal_spread(self, tier):
        """Devuelve s* para el tier dado."""
        ...

    def summary(self):
        """Imprime tabla de coeficientes y s* por tier."""
        ...
```

Usa la clase en el notebook: instancia `model = RFQModel()`, entrénala con `df_model`, y llama a `model.summary()`.

In [ ]:
# Tu código aquí (implementa la clase en este notebook o en rfq_model.py)
pass

In [ ]:
# ── Validación E10 ─────────────────────────────────────────────────
assert 'RFQModel' in dir() or 'RFQModel' in globals(), \
    "Define la clase RFQModel en este notebook"
model = RFQModel()
model.fit(df_model)
p = model.predict_close_probability(1, 0.5)
assert abs(p - 0.14983) < 0.001, f"predict_close_probability(1, 0.5): esperado 0.14983, tienes {p:.5f}"
s = model.optimal_spread(1)
assert abs(s - 0.9484) < 0.01, f"optimal_spread(1): esperado 0.9484, tienes {s:.4f}"
print("✓ E10 correcto — RFQModel funciona")
model.summary()

In [ ]:
# ── Solución E10 ──────────────────────────────────────────────────
class RFQModel:
    def __init__(self):
        self._params = {}  # tier -> (b0, b1)

    def fit(self, df_model):
        self._params = {}
        for t in [1, 2, 3]:
            mask = df_model['tier'] == t
            X = df_model.loc[mask, 'spread_bp'].values.reshape(-1, 1)
            y = df_model.loc[mask, 'won'].values
            lr = LogisticRegression(random_state=42)
            lr.fit(X, y)
            self._params[t] = (lr.intercept_[0], lr.coef_[0][0])
        return self

    def predict_close_probability(self, tier, spread_bp):
        b0, b1 = self._params[tier]
        return float(expit(b0 + b1 * spread_bp))

    def expected_revenue(self, tier, spread_bp, Q=1.0):
        return self.predict_close_probability(tier, spread_bp) * spread_bp * Q

    def optimal_spread(self, tier, Q=1.0):
        result = minimize_scalar(
            lambda s: -self.expected_revenue(tier, s, Q),
            bounds=(0.01, 5.0),
            method='bounded'
        )
        return result.x

    def summary(self):
        print(f"{'Tier':>5} {'b0':>10} {'b1':>10} {'s*':>10} {'P(s*)':>8}")
        print('-' * 50)
        for t in [1, 2, 3]:
            b0, b1 = self._params[t]
            s = self.optimal_spread(t)
            p = self.predict_close_probability(t, s)
            print(f"  T{t}   {b0:>10.4f} {b1:>10.4f} {s:>10.4f} {p:>7.2%}")

model = RFQModel()
model.fit(df_model)
model.summary()